# Calibration Comparison: Energy vs Campaign-Time
#
# This notebook compares the two calibration methods on all datasets
# to justify using campaign-time calibration for experiments.

In [ ]:
import sys
import os

# Ensure repo root is on sys.path regardless of kernel cwd
_notebook_dir = os.path.dirname(os.path.abspath("__file__"))
_repo_root = os.path.abspath(os.path.join(_notebook_dir, "..")) if os.path.basename(_notebook_dir) == "experiments" else _notebook_dir
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)
os.chdir(_repo_root)

from uav_routing.environment import Environment
from uav_routing.environment.calibration import calibrate, calibration_info
from uav_routing.environment.graph import Graph
from uav_routing.environment.drone import Drone
from uav_routing.solver.exact import solve_model_gurobi, print_table
import gurobipy as gp
import pandas as pd

# Dataset paths (relative to repo root)
datasets = {
    "C101 (50)":  "datasets/data/50_c101.txt",
    "R101 (50)":  "datasets/data/50_r101.txt",
    "RC101 (100)": "datasets/data/rc101.txt",
    "R101 (100)": "datasets/data/r101.txt",
    "C101 (100)": "datasets/data/c101.txt",
}

seed = 42
time_limit = 600
sortie_time = 3.0

## 1. Calibration Summary per Dataset

Show how spatial scale, time scale, and energy budget differ between the two methods.

In [2]:
rows = []
for name, path in datasets.items():
    for method in ["energy", "campaign"]:
        graph = Graph(path=path, seed=1)
        uav = Drone(base=graph.graph['base'])
        calib = calibrate(
            graph=graph,
            tour_length=len(graph.nodes) // 2,
            drone=uav,
            drone_sortie_time=sortie_time,
            calibration_method=method,
        )
        solomon_campaign = graph.nodes[uav.base]['time_window'][1]
        rows.append({
            "Instance": name,
            "Method": method,
            "Nodes": len(graph.nodes),
            "Solomon c_max": solomon_campaign,
            "Spatial scale (m/unit)": round(calib.spatial_scale, 2),
            "Time scale (s/unit)": round(calib.time_scale, 4),
            "Scaled campaign (s)": round(calib.graph.nodes[uav.base]['time_window'][1], 1),
            "E_max (J)": f"{calib.scaled_max_energy:.2e}",
            "Ref tour dist (Solomon)": round(calib.tour_dist_solomon, 1),
        })

df_calib = pd.DataFrame(rows)
df_calib

,Instance,Method,Nodes,Solomon c_max,Spatial scale (m/unit),Time scale (s/unit),Scaled campaign (s),E_max (J),Ref tour dist (Solomon)
0,C101 (50),energy,51,1236.0,575.11,12.9793,16042.4,1.70e+08,832.1
1,C101 (50),campaign,51,1236.0,387.17,8.7379,10800.0,1.70e+08,135.5
2,R101 (50),energy,51,230.0,549.40,12.3990,2851.8,1.70e+08,871.0
3,R101 (50),campaign,51,230.0,2080.64,46.9565,10800.0,1.70e+08,223.3
4,RC101 (100),energy,101,240.0,195.87,4.4205,1060.9,1.70e+08,2443.2
5,RC101 (100),campaign,101,240.0,1993.95,45.0000,10800.0,1.70e+08,384.6
6,R101 (100),energy,101,230.0,270.35,6.1013,1403.3,1.70e+08,1770.1
7,R101 (100),campaign,101,230.0,2080.64,46.9565,10800.0,1.70e+08,365.2
8,C101 (100),energy,101,1236.0,208.04,4.6952,5803.3,1.70e+08,2300.2
9,C101 (100),campaign,101,1236.0,387.17,8.7379,10800.0,1.70e+08,243.8


## 2. Solve with Both Calibrations

Run Gurobi on all datasets with both methods and compare objective, gap, solve time, tour length.

In [3]:
results_all = {}

with gp.Env(empty=True) as env:
    env.setParam('OutputFlag', 0)
    env.start()

    for name, path in datasets.items():
        for method in ["energy", "campaign"]:
            label = f"{name} | {method}"
            print(f"Solving: {label}...")

            graph = Graph(path=path, seed=1)
            uav = Drone(base=graph.graph['base'])
            calib = calibrate(
                graph=graph,
                tour_length=len(graph.nodes) // 2,
                drone=uav,
                drone_sortie_time=sortie_time,
                calibration_method=method,
            )
            instance = Environment(calib, uav)

            res = solve_model_gurobi(instance, seed=seed, time_limit=time_limit,
                                     env=env, prune=False, stats=False)

            results_all[label] = {
                "dataset": name,
                "method": method,
                "nodes": len(graph.nodes),
                "obj": res.get("obj") if res else None,
                "gap": res.get("gap") if res else None,
                "solve_time": res.get("solve_time") if res else None,
                "tour_len": len(res.get("tour", [])) - 2 if res and res.get("tour") else None,
                "spatial_scale": calib.spatial_scale,
                "time_scale": calib.time_scale,
                "E_max": calib.scaled_max_energy,
                "scaled_campaign": calib.graph.nodes[uav.base]['time_window'][1],
            }

print("Done.")

Solving: C101 (50) | energy...


Solving: C101 (50) | campaign...


Solving: R101 (50) | energy...


Solving: R101 (50) | campaign...


Solving: RC101 (100) | energy...


...

Solving: R101 (100) | energy...


Solving: R101 (100) | campaign...


Solving: C101 (100) | energy...


Solving: C101 (100) | campaign...


Done.


In [4]:
# Summary table
df = pd.DataFrame(results_all).T
df.index.name = "Instance | Calibration"

df_display = df.copy()
df_display["obj"] = df_display["obj"].apply(lambda x: f"{x:.2f}" if x is not None else "INFEASIBLE")
df_display["gap"] = df_display["gap"].apply(lambda x: f"{x:.4%}" if x is not None else "-")
df_display["solve_time"] = df_display["solve_time"].apply(lambda x: f"{x:.1f}s" if x is not None else "-")
df_display["spatial_scale"] = df_display["spatial_scale"].apply(lambda x: f"{x:.2f}")
df_display["time_scale"] = df_display["time_scale"].apply(lambda x: f"{x:.4f}")
df_display["E_max"] = df_display["E_max"].apply(lambda x: f"{x:.2e}")
df_display["scaled_campaign"] = df_display["scaled_campaign"].apply(lambda x: f"{x/3600:.2f}h")

print("COMPARISON: ENERGY vs CAMPAIGN CALIBRATION")
print("=" * 100)
print(df_display[["dataset", "method", "nodes", "obj", "gap", "solve_time",
                   "tour_len", "spatial_scale", "time_scale", "E_max", "scaled_campaign"]].to_string())

COMPARISON: ENERGY vs CAMPAIGN CALIBRATION
                            dataset    method nodes        obj        gap solve_time tour_len spatial_scale time_scale     E_max scaled_campaign
Instance | Calibration                                                                                                                          
C101 (50) | energy        C101 (50)    energy    51   24581.82    0.0000%      10.7s       21        575.11    12.9793  1.70e+08           4.46h
C101 (50) | campaign      C101 (50)  campaign    51   14669.20    0.0000%       4.1s       33        387.17     8.7379  1.70e+08           3.00h
R101 (50) | energy        R101 (50)    energy    51   10184.28    0.0010%       3.0s        7        549.40    12.3990  1.70e+08           0.79h
R101 (50) | campaign      R101 (50)  campaign    51  144232.37    0.0000%       2.3s        6       2080.64    46.9565  1.70e+08           3.00h
RC101 (100) | energy    RC101 (100)    energy   101    2475.80  177.7855%     600.1s   

## 3. Key Observations

**Why campaign-time calibration is preferred:**

1. **Time-window feasibility is guaranteed.** Campaign-time calibration maps Solomon's depot deadline directly to `T_sortie`, so scaled time windows are always physically meaningful. Energy-based calibration derives `beta` from the reference tour distance, which may produce a scaled campaign time that is not `T_sortie`.

2. **Instance-independent energy budget.** Under campaign-time calibration, `E_max = P(v_opt) * T_sortie` — the same for all instances. This makes cross-instance comparison meaningful. Under energy-based calibration, `E_max` depends on the reference tour geometry, so different instances get different energy budgets.

3. **Clean separation of concerns.** The `eta` parameter scales energy independently: `eta=1` means full sortie energy, `eta<1` tightens it. This lets us study the energy-time trade-off cleanly without touching the time structure.

4. **Objectives are comparable.** Since spatial and temporal scales are deterministic functions of `c_max` and `v_opt`, the information values (which depend on slopes * time) are on the same physical scale across instances with the same `c_max`.

**When energy-based calibration might be preferred:**

- If the goal is to make the energy constraint binding by construction (the reference tour exactly exhausts the budget). This is useful for studying energy-constrained regimes without tuning `eta`.
- For instances where the campaign time is very large relative to the spatial spread, campaign-time calibration may produce very large spatial scales, making distances unrealistically large.

**Hybrid approach: campaign-time scaling with energy-based budget**

A hybrid calibration takes the best of both methods:

1. **Use campaign-time for scales.** Set `β = T_sortie / c_max` and `α = β · v_opt`, exactly as in campaign-time calibration. This guarantees TW feasibility and instance-independent spatial/temporal scaling.

2. **Use energy-based method for `E_max`.** Compute the reference tour in scaled coordinates, then set `E_max = E(v_opt, d_R)` where `d_R` is the scaled reference tour distance. This makes the energy constraint binding by construction for the reference tour.

**Why this helps:** Under pure campaign-time calibration, `E_max = P(v_opt) · T_sortie` represents the *full sortie energy* — the drone could fly for 3 hours straight at optimal speed. For spatially compact instances (e.g., C101 with clustered nodes), this budget is far more than any feasible tour needs, making the energy constraint inactive. The hybrid fixes this: the energy budget is calibrated to the instance geometry (tight enough to be active), while the time structure remains clean and comparable.

**Trade-off:** `E_max` becomes instance-dependent again (different reference tour distances → different budgets), so cross-instance energy comparisons require normalization. However, the `η` parameter still works cleanly: `η = 1` means the reference tour is exactly affordable, `η < 1` means only a fraction of that energy is available. This is arguably more interpretable than campaign-time's `η`, where `η = 1` may be far from binding.

**When to use which:**
| | Campaign-time | Energy-based | Hybrid |
|---|---|---|---|
| TW feasibility | ✓ guaranteed | ✗ may violate | ✓ guaranteed |
| E_max instance-independent | ✓ | ✗ | ✗ |
| Energy constraint active | only if η tuned | ✓ by construction | ✓ by construction |
| Cross-instance comparability | ✓ direct | ✗ | ✓ after η-normalization |
| Best for | controlled experiments | single-instance analysis | balanced experiments |

## 4. Hybrid Calibration Results

Campaign-time scaling (β, α) with energy-based E_max from reference tour.

In [4]:
# Step 1: Campaign calibration gives scaled data + E_max = P(v_opt) * T_sortie
# Step 2: Compute NN reference tour on scaled graph, get tour energy
# Step 3: Replace E_max with tour energy, solve

hybrid_results = {}
hybrid_table_rows = []

with gp.Env(empty=True) as env:
    env.setParam('OutputFlag', 0)
    env.start()

    for name, path in datasets.items():
        print(f"Solving: {name} | hybrid...")

        # Campaign calibration for scaling
        graph = Graph(path=path, seed=1)
        uav = Drone(base=graph.graph['base'])
        campaign_calib = calibrate(
            graph=graph,
            tour_length=len(graph.nodes) // 2,
            drone=uav,
            drone_sortie_time=sortie_time,
            calibration_method="campaign",
        )
        campaign_emax = campaign_calib.scaled_max_energy

        # Hybrid: same scaling, E_max from reference tour
        graph2 = Graph(path=path, seed=1)
        hybrid_calib = calibrate(
            graph=graph2,
            tour_length=len(graph2.nodes) // 2,
            drone=uav,
            drone_sortie_time=sortie_time,
            calibration_method="hybrid",
        )
        ref_tour_energy = hybrid_calib.scaled_max_energy

        # Solve with replaced E_max
        instance = Environment(hybrid_calib, uav)
        res = solve_model_gurobi(instance, seed=seed, time_limit=time_limit,
                                 env=env, prune=False, stats=False)

        hybrid_table_rows.append({
            "Instance": name,
            "Nodes": len(graph.nodes),
            "Campaign E_max (J)": f"{campaign_emax:.2e}",
            "Ref tour energy (J)": f"{ref_tour_energy:.2e}",
            "Ratio (tour/campaign)": f"{ref_tour_energy / campaign_emax:.2%}",
            "Obj": f"{res['obj']:.2f}" if res and res.get('obj') else "INFEASIBLE",
            "Gap": f"{res['gap']:.4%}" if res and res.get('gap') is not None else "-",
            "Time": f"{res['solve_time']:.1f}s" if res and res.get('solve_time') else "-",
            "Tour len": len(res.get("tour", [])) - 2 if res and res.get("tour") else "-",
        })

print("Done.\n")
df_hybrid = pd.DataFrame(hybrid_table_rows)
df_hybrid

Solving: C101 (50) | hybrid...
Solving: R101 (50) | hybrid...
Solving: RC101 (100) | hybrid...
Solving: R101 (100) | hybrid...
Solving: C101 (100) | hybrid...
Done.



,Instance,Nodes,Campaign E_max (J),Ref tour energy (J),Ratio (tour/campaign),Obj,Gap,Time,Tour len
0,C101 (50),51,1.70e+08,1.87e+07,10.96%,518.51,0.0040%,57.0s,3
1,R101 (50),51,1.70e+08,1.65e+08,97.08%,144232.37,0.0000%,2.4s,6
2,RC101 (100),101,1.70e+08,2.73e+08,160.25%,253357.12,98.5311%,600.1s,12
3,R101 (100),101,1.70e+08,2.70e+08,158.78%,306304.51,0.0000%,31.6s,12
4,C101 (100),101,1.70e+08,3.36e+07,19.73%,4542.57,161.9614%,600.0s,8


## 5. Uniform Calibration Results

Like energy calibration but with a single scaler β for both distance and time (α = β).
This preserves Solomon's d = t assumption while scaling to real-world units.
β = T_sortie / tour_dist_solomon. E_max from reference tour energy at v_opt.

In [3]:
uniform_rows = []

with gp.Env(empty=True) as env:
    env.setParam('OutputFlag', 0)
    env.start()

    for name, path in datasets.items():
        print(f"Solving: {name} | uniform...")

        graph = Graph(path=path, seed=1)
        uav = Drone(base=graph.graph['base'])
        calib = calibrate(
            graph=graph,
            tour_length=len(graph.nodes) // 2,
            drone=uav,
            drone_sortie_time=sortie_time,
            calibration_method="uniform",
        )
        instance = Environment(calib, uav)
        res = solve_model_gurobi(instance, seed=seed, time_limit=time_limit,
                                 env=env, prune=False, stats=False)

        uniform_rows.append({
            "Instance": name,
            "Nodes": len(graph.nodes),
            "Spatial scale": round(calib.spatial_scale, 4),
            "Time scale": round(calib.time_scale, 4),
            "α = β": "✓" if abs(calib.spatial_scale - calib.time_scale) < 1e-6 else "✗",
            "Scaled campaign (s)": round(calib.graph.nodes[uav.base]['time_window'][1], 1),
            "E_max (J)": f"{calib.scaled_max_energy:.2e}",
            "Obj": f"{res['obj']:.2f}" if res and res.get('obj') else "INFEASIBLE",
            "Gap": f"{res['gap']:.4%}" if res and res.get('gap') is not None else "-",
            "Time": f"{res['solve_time']:.1f}s" if res and res.get('solve_time') else "-",
            "Tour len": len(res.get("tour", [])) - 2 if res and res.get("tour") else "-",
        })
        if res and res.get("tour"):
            print_table(instance, res)

print("Done.\n")
df_uniform = pd.DataFrame(uniform_rows)
df_uniform

Solving: C101 (50) | uniform...

--- Gurobi Solver Report ---
      Edge  Total Time (s)  Loiter Time (s)  Speed (m/s)  SOCP Energy (J)  Real Energy (J)  Energy %  Arrival Time (s)     Time Window     Slope
     0->20          129.79           125.95        33.81       1794112.53       1794117.60   46.7306            129.79  [129.8, 947.5] -1.955360
     20->5           64.90            58.99        42.20       1573988.96        978142.01   25.4772            194.69  [194.7, 869.6] -0.252143
     5->43           12.98             2.92        40.41        189583.03        189627.94    4.9392            207.67 [207.7, 1038.3] -0.678594
     43->0            8.28             0.00        25.94        281595.84        124544.93    3.2440            215.95  [0.0, 16042.4]  0.000000
TOTAL TOUR          215.95           187.86          NaN       3839280.35       3086432.48  100.0000               NaN             NaN       NaN
------------------------------------------------------------
Objecti

,Instance,Nodes,Spatial scale,Time scale,α = β,Scaled campaign (s),E_max (J),Obj,Gap,Time,Tour len
0,C101 (50),51,12.9793,12.9793,✓,16042.4,3.84e+06,30.00,35021.9174%,600.0s,3
1,R101 (50),51,12.3990,12.3990,✓,2851.8,3.84e+06,INFEASIBLE,-,-,-
2,RC101 (100),101,4.4205,4.4205,✓,1060.9,3.84e+06,1016.96,866.5953%,601.8s,24
3,R101 (100),101,6.1013,6.1013,✓,1403.3,3.84e+06,868.94,1414.7024%,600.0s,13
4,C101 (100),101,4.6952,4.6952,✓,5803.3,3.84e+06,441.16,2553.2275%,600.0s,10
